# ProductAnalytics_Build

Run through the `ProductAnalytics_Process` or `ProductAnalytics_Demo` pipeline, or interactively with cells in order. Deployment never executes this notebook.


In [ ]:
# Pipelines inject these values from ProductAnalytics_Config; keep None interactively.
allow_synthetic_overwrite = None
bronze_id = None
data_workspace_id = None
identity_id = None
pii_model = None
pii_policy_version = None
silver_id = None
sources_json = None
vault_workspace_id = None


## Configuration and validation

Pipeline runs pass every `ProductAnalytics_Config` value as a parameter; interactive runs read the active value set directly. Validate environment IDs, the source registry and PII settings before any data access. Missing, partial or example configuration fails explicitly; no fallback is used.


In [ ]:
"""Validated runtime configuration; no Fabric access during module import."""

import json
import re
import uuid

LIBRARY_NAME = "ProductAnalytics_Config"
ID_FIELDS = ("data_workspace_id", "vault_workspace_id", "bronze_id", "silver_id", "identity_id")
SETTING_FIELDS = {"sources_json", "pii_policy_version", "pii_model", "allow_synthetic_overwrite"}
FIELDS = set(ID_FIELDS) | SETTING_FIELDS
SCHEMA = "product"
TABLE_NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_]*\Z")


def onelake_table(workspace_id, lakehouse_id, table):
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{SCHEMA}/{table}"


def onelake_files(workspace_id, lakehouse_id, path):
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/{SCHEMA}/{path}"


def validate_config(values):
    if set(values) != FIELDS:
        raise ValueError("Runtime configuration fields do not match the contract")
    config = dict(values)
    for name in ID_FIELDS:
        value = values[name]
        if not isinstance(value, str):
            raise ValueError(f"{name} must be a GUID string")
        value = str(uuid.UUID(value))
        if value.startswith("00000000-"):
            raise ValueError("Example configuration only; bind a real environment before execution")
        config[name] = value
    if len({config[name] for name in ("bronze_id", "silver_id", "identity_id")}) != 3:
        raise ValueError("Bronze, Silver and Identity must be distinct lakehouses")
    # Re-identification data stays in a separate workspace (GDPR Art. 4(5)); see docs/architecture.md.
    if config["data_workspace_id"] == config["vault_workspace_id"]:
        raise ValueError("Data and Vault must be distinct workspaces")
    for name in ("pii_policy_version", "pii_model"):
        if not isinstance(values[name], str) or not values[name].strip():
            raise ValueError(f"{name} must be a nonempty string")
    if type(values["allow_synthetic_overwrite"]) is not bool:
        raise ValueError("allow_synthetic_overwrite must be a Boolean")
    if not isinstance(values["sources_json"], str):
        raise ValueError("sources_json must be a JSON string")
    sources = json.loads(values["sources_json"])
    if not isinstance(sources, list) or not sources:
        raise ValueError("At least one source is required")
    tenants, tables = set(), set()
    for source in sources:
        if not isinstance(source, dict) or set(source) != {"tenant_id", "users_table", "events_table"}:
            raise ValueError("Each source must define tenant_id, users_table and events_table")
        tenant = source["tenant_id"]
        if not isinstance(tenant, str) or not TABLE_NAME.fullmatch(tenant) or tenant in tenants:
            raise ValueError("Source tenant IDs must be unique safe identifiers")
        tenants.add(tenant)
        for name in ("users_table", "events_table"):
            table = source[name]
            if not isinstance(table, str) or not TABLE_NAME.fullmatch(table) or table in tables:
                raise ValueError("Source table names must be unique safe identifiers")
            tables.add(table)
    config["sources"] = sources
    return config


def load_runtime_config(notebook_utils, parameters=None):
    """Pipeline runs inject every field from the library; interactive runs read it directly.

    Service principals cannot read Variable Libraries from NotebookUtils, so pipelines
    resolve the values and pass them as notebook parameters.
    """
    provided = {name: value for name, value in (parameters or {}).items() if value is not None}
    if provided:
        if set(provided) != FIELDS:
            raise ValueError("Pipeline parameters must supply every configuration field")
        return validate_config(provided)
    library = notebook_utils.variableLibrary.getLibrary(LIBRARY_NAME)
    values = {name: library.getVariable(name) for name in sorted(FIELDS)}
    return validate_config(values)

import notebookutils
CONFIG = load_runtime_config(notebookutils, {name: globals().get(name) for name in FIELDS})
print("Runtime configuration validated; values are not printed.")


## Versioned synthetic fixture

Define the deterministic English-only A/B/C test fixture. This is test data, not an ingestion connector. Source table names come from the validated registry.


In [ ]:
import json
import random
from datetime import datetime, timedelta
DATA_WORKSPACE_ID = CONFIG["data_workspace_id"]
BRONZE_ID = CONFIG["bronze_id"]
TENANTS = [('tenant_a', 'Example Company A', 'ES', 12), ('tenant_b', 'Example Company B', 'UK', 18), ('tenant_c', 'Example Company C', 'DE', 24)]
FEATURES = [('news', 'News', 'Communications'), ('search', 'Search', 'Discovery'), ('knowledge', 'Knowledge', 'Collaboration'), ('community', 'Community', 'Collaboration')]
FIXTURE_VERSION = 'abc-english-v2'
RAW_TEXT_CASES = {'en': ['Contact Alex Morgan at user001@{domain}.example.invalid.', 'Contact phone: +1 202-555-0147.', 'Send to user001@{domain}.example.invalid; copy user001@{domain}.example.invalid.', 'Search does not return the help article.', 'Please update my employee record. I live at 42 Example Road, London SW1A 1AA. My UK passport number is 123456789 and my National Insurance number is QQ123456C.', 'Please update my details. My address is Musterstrasse 17, 10115 Berlin. My German national identity number is L01X00T47 and my passport number is C01X00T47.', 'Please update my record. My address is 18 rue Exemple, 75001 Paris. My French social security number is 1 85 05 75 123 456 78 and my passport number is 12AB34567.', 'I moved to 24 Voorbeeldstraat, 1012 AB Amsterdam. My Dutch BSN is 123456782. Please use my passport number NX1234567 when updating the travel request.', 'Please update my profile. My address is Calle Ejemplo 25, 28013 Madrid. My Spanish national identity number is 12345678Z and my passport number is AAA123456. The ticket reference is INC-48271.', '', None]}

def add_raw_text(events):
    language_by_tenant = {tenant[0]: "en" for tenant in TENANTS}
    counts = {tenant: 0 for tenant in language_by_tenant}
    result = []
    for event in events:
        tenant = event[1]
        language = language_by_tenant[tenant]
        text, text_language = None, None
        if event[4].startswith("2026-09"):
            index = counts[tenant]
            if index < len(RAW_TEXT_CASES[language]):
                template = RAW_TEXT_CASES[language][index]
                text = template.format(domain=tenant.replace("_", "-")) if template is not None else None
                text_language = language
            counts[tenant] += 1
        result.append((*event, text, text_language))
    return result

def generate_data():
    rng = random.Random(20261002)
    users, events = [], []
    for tenant_index, (tenant_id, _, _, user_count) in enumerate(TENANTS):
        for user_index in range(1, user_count + 1):
            source_id = f"user{user_index:03d}"
            # Preserve the original event fixture's RNG sequence without generating identity keys.
            rng.getrandbits(128)
            name = "Alex Morgan" if user_index == 1 else f"Test User {user_index:03d}"
            email = f"{source_id}@{tenant_id.replace('_', '-')}.example.invalid"
            users.append((tenant_id, source_id, name, email))
            for month in (8, 9):
                for day in range(1, 29):
                    for repetition in range(1 + (user_index + day + tenant_index) % 3):
                        feature = FEATURES[(user_index + day + repetition) % len(FEATURES)][0]
                        timestamp = datetime(2026, month, day, 9) + timedelta(minutes=user_index + repetition)
                        events.append((
                            f"event{len(events) + 1:06d}", tenant_id, source_id, feature,
                            timestamp.isoformat(), "view", 10 + rng.randrange(291),
                        ))
    return users, add_raw_text(events)

def split_source_rows(rows, tenant_index):
    partitions = {t[0]: [] for t in TENANTS}
    for row in rows:
        tenant = row[tenant_index]
        if tenant not in partitions:
            raise ValueError(f"Unknown tenant: {tenant}")
        partitions[tenant].append(row)
    return partitions

def raw_load_mode(config, existing_tables):
    """Initial load into empty Bronze is allowed; replacing existing RAW needs the explicit flag."""
    if {source["tenant_id"] for source in config["sources"]} != {tenant[0] for tenant in TENANTS}:
        raise RuntimeError("The synthetic generator supports only the versioned A/B/C fixture")
    if not existing_tables:
        return "initial_load"
    if not config["allow_synthetic_overwrite"]:
        raise RuntimeError(
            "RAW tables already exist and synthetic overwrite is disabled in runtime configuration: "
            + ", ".join(sorted(existing_tables)))
    return "overwrite"


## RAW publication and summary

Requires the exact A/B/C tenant set. Loads empty Bronze `product` tables once; replacing existing RAW requires explicit `allow_synthetic_overwrite=true`. Writes synthetic RAW inputs only; never run as part of deployment.


In [ ]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def location(table):
    return onelake_table(DATA_WORKSPACE_ID, BRONZE_ID, table)

raw_tables = [source[name] for source in CONFIG["sources"] for name in ("users_table", "events_table")]
LOAD_MODE = raw_load_mode(CONFIG, [t for t in raw_tables if DeltaTable.isDeltaTable(spark, location(t))])
users, event_rows = generate_data()

user_schema = "tenant_id string, source_user_id string, display_name string, email string"
event_schema = "event_id string, tenant_id string, source_user_id string, feature_id string, occurred_at string, event_type string, duration_seconds long, free_text string, text_language string"
user_sources = split_source_rows(users, 0)
event_sources = split_source_rows(event_rows, 1)
bronze_evidence = {}
for tenant_id, _, _, _ in TENANTS:
    source = next(source for source in CONFIG["sources"] if source["tenant_id"] == tenant_id)
    raw_users = spark.createDataFrame(user_sources[tenant_id], user_schema)
    raw_events = spark.createDataFrame(event_sources[tenant_id], event_schema)
    bronze_evidence[tenant_id] = {}
    for name, df in (("users", raw_users), ("events", raw_events)):
        landing = onelake_files(DATA_WORKSPACE_ID, BRONZE_ID, f"landing/{tenant_id}/{name}")
        df.write.mode("overwrite").json(landing)
        landed = spark.read.schema(df.schema).json(landing)
        assert landed.filter(F.col("tenant_id") != tenant_id).count() == 0
        path = location(source["users_table"] if name == "users" else source["events_table"])
        # Additive fixture schema evolution only; do not replace existing schemas.
        landed.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(path)
        persisted = spark.read.format("delta").load(path)
        assert persisted.exceptAll(df).limit(1).count() == 0
        assert df.exceptAll(persisted).limit(1).count() == 0
        bronze_evidence[tenant_id][name] = persisted.count()
    bronze_evidence[tenant_id]["text_cases"] = [
        row.asDict() for row in spark.read.format("delta").load(
            location(source["events_table"])
        ).filter(F.col("text_language").isNotNull()).select(
            "event_id", "free_text", "text_language"
        ).orderBy("event_id").collect()
    ]

evidence = {
    "purpose": "Synthetic raw inputs only; no downstream processing",
    "load_mode": LOAD_MODE,
    "synthetic_only": True,
    "fixture_version": FIXTURE_VERSION,
    "raw_events": len(event_rows),
    "raw_users": len(users),
    "bronze_separate_sources": bronze_evidence,
}
spark.createDataFrame([(json.dumps(evidence, sort_keys=True),)], "value string").coalesce(1).write.mode("overwrite").text(
    onelake_files(DATA_WORKSPACE_ID, BRONZE_ID, f"validation/{FIXTURE_VERSION}")
)
print(json.dumps(evidence, indent=2))
